In [1]:
import pandas as pd
import numpy as np

from scipy.sparse import hstack

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [ ]:
dev_df = pd.read_csv("../preprocessing/clean_data_with_clean_target.csv")


required_cols = ["prev_clean", "target_clean", "label_id", "group"]

assert all(c in dev_df.columns for c in required_cols)
# assert all(c in test_df.columns for c in ["prev_clean", "target_clean", "label_id"])

dev_df = dev_df.reset_index(drop=True)
# test_df = test_df.reset_index(drop=True)

print("Development dataset:", dev_df.shape)
# print("Separate test dataset:", test_df.shape)
print("Development classes:", sorted(dev_df["label_id"].unique()))

Development dataset: (19431, 10)
Development classes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]


In [4]:
splitter = StratifiedGroupKFold(
  n_splits=10,
  shuffle=True,
  random_state=42
)

dev_df["fold"] = -1

for fold_number, (_, val_indices) in enumerate(
  splitter.split(
    X=dev_df,
    y=dev_df["label_id"],
    groups=dev_df["group"]
  )
):
  dev_df.loc[val_indices, "fold"] = fold_number

print(dev_df["fold"].value_counts().sort_index())

fold
0    1887
1    2111
2    1970
3    2000
4    1851
5    1862
6    1987
7    1953
8    1883
9    1927
Name: count, dtype: int64


In [5]:
def build_features(train_df, val_df):
  # Extract context and target text
  train_target = train_df["target_clean"].fillna("").astype(str)
  train_prev = train_df["prev_clean"].fillna("").astype(str)

  val_target = val_df["target_clean"].fillna("").astype(str)
  val_prev = val_df["prev_clean"].fillna("").astype(str)

  # Target word features
  target_word = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
    dtype=np.float32
  )

  # Target character features
  target_char = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 5),
    min_df=2,
    max_features=100_000,
    sublinear_tf=True,
    dtype=np.float32
  )

  # Context word features
  prev_word = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
    dtype=np.float32
  )

  # Fit vectorizers only on the current training fold
  X_train_tw = target_word.fit_transform(train_target)
  X_train_tc = target_char.fit_transform(train_target)
  X_train_pw = prev_word.fit_transform(train_prev)

  # Transform validation fold using training vocabularies
  X_val_tw = target_word.transform(val_target)
  X_val_tc = target_char.transform(val_target)
  X_val_pw = prev_word.transform(val_prev)

  # Join features in the same order
  X_train = hstack(
    [X_train_tw, X_train_tc, X_train_pw],
    format="csr"
  )

  X_val = hstack(
    [X_val_tw, X_val_tc, X_val_pw],
    format="csr"
  )

  return X_train, X_val

In [6]:
fold_results = []

for fold_number in range(10):
  print(f"\n========== Fold {fold_number} ==========")

  train_fold = dev_df[dev_df["fold"] != fold_number]
  val_fold = dev_df[dev_df["fold"] == fold_number]

  # Separate labels
  y_train = train_fold["label_id"]
  y_val = val_fold["label_id"]

  # Build features for this fold
  X_train, X_val = build_features(train_fold, val_fold)

  # Train Logistic Regression
  model = LogisticRegression(
    C=1.0,
    class_weight="balanced",
    max_iter=2000,
    solver="lbfgs",
    random_state=42
  )

  model.fit(X_train, y_train)

  # Validate
  val_pred = model.predict(X_val)

  macro_f1 = f1_score(
    y_val, val_pred, average="macro", zero_division=0
  )

  weighted_f1 = f1_score(
    y_val, val_pred, average="weighted", zero_division=0
  )

  accuracy = accuracy_score(y_val, val_pred)

  fold_results.append({
    "fold": fold_number,
    "macro_f1": macro_f1,
    "weighted_f1": weighted_f1,
    "accuracy": accuracy
  })

  print("Training rows:", len(train_fold))
  print("Validation rows:", len(val_fold))
  print(f"Macro F1:    {macro_f1:.4f}")
  print(f"Weighted F1: {weighted_f1:.4f}")
  print(f"Accuracy:    {accuracy:.4f}")

  # Release fold-specific objects
  del X_train, X_val, model


========== Fold 0 ==========
Training rows: 17544
Validation rows: 1887
Macro F1:    0.9647
Weighted F1: 0.9773
Accuracy:    0.9772

========== Fold 1 ==========
Training rows: 17320
Validation rows: 2111
Macro F1:    0.9517
Weighted F1: 0.9712
Accuracy:    0.9711

========== Fold 2 ==========
Training rows: 17461
Validation rows: 1970
Macro F1:    0.9757
Weighted F1: 0.9853
Accuracy:    0.9853

========== Fold 3 ==========
Training rows: 17431
Validation rows: 2000
Macro F1:    0.9763
Weighted F1: 0.9821
Accuracy:    0.9820

========== Fold 4 ==========
Training rows: 17580
Validation rows: 1851
Macro F1:    0.9656
Weighted F1: 0.9765
Accuracy:    0.9762

========== Fold 5 ==========
Training rows: 17569
Validation rows: 1862
Macro F1:    0.9737
Weighted F1: 0.9808
Accuracy:    0.9807

========== Fold 6 ==========
Training rows: 17444
Validation rows: 1987
Macro F1:    0.9579
Weighted F1: 0.9707
Accuracy:    0.9703

========== Fold 7 ==========
Training rows: 17478
Validation rows: 1

In [7]:
results_df = pd.DataFrame(fold_results)

display(results_df)

print("\nCross-validation summary:")
print(
  results_df[["macro_f1", "weighted_f1", "accuracy"]]
  .agg(["mean", "std"])
)

,fold,macro_f1,weighted_f1,accuracy
0,0,0.964712,0.977331,0.977213
1,1,0.951669,0.971157,0.971104
2,2,0.975747,0.985337,0.985279
3,3,0.976252,0.982136,0.982000
4,4,0.965627,0.976456,0.976229
5,5,0.973684,0.980800,0.980666
6,6,0.957936,0.970688,0.970307
7,7,0.963931,0.972738,0.972862
8,8,0.970502,0.977921,0.977695
9,9,0.960559,0.978000,0.977686



Cross-validation summary:
      macro_f1  weighted_f1  accuracy
mean  0.966062     0.977256  0.977104
std   0.008065     0.004762  0.004774


In [8]:
# Prepare text
X_dev_target = dev_df["target_clean"].fillna("").astype(str)
X_dev_prev = dev_df["prev_clean"].fillna("").astype(str)

test_df = pd.read_csv("../preprocessing/clean_test_data_with_preprocessing.csv")  # Change to your actual filename

X_test_target = test_df["reply_clean"].fillna("").astype(str)
X_test_prev = test_df["context_clean"].fillna("").astype(str)

# Fit the three vectorizers on all development data
tfidf_target_word = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
    dtype=np.float32
)

tfidf_target_char = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 5),
    min_df=2,
    max_features=100_000,
    sublinear_tf=True,
    dtype=np.float32
)

tfidf_prev_word = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
    dtype=np.float32
)

X_dev_tw = tfidf_target_word.fit_transform(X_dev_target)
X_dev_tc = tfidf_target_char.fit_transform(X_dev_target)
X_dev_pw = tfidf_prev_word.fit_transform(X_dev_prev)

# Transform the separate test set
X_test_tw = tfidf_target_word.transform(X_test_target)
X_test_tc = tfidf_target_char.transform(X_test_target)
X_test_pw = tfidf_prev_word.transform(X_test_prev)

# Combine features
X_dev = hstack([X_dev_tw, X_dev_tc, X_dev_pw], format="csr")
X_test = hstack([X_test_tw, X_test_tc, X_test_pw], format="csr")

# Train final model
final_model = LogisticRegression(
    C=1.0,
    class_weight="balanced",
    max_iter=2000,
    solver="lbfgs",
    random_state=42
)

final_model.fit(X_dev, dev_df["label_id"])

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'lbfgs'
,max_iter,2000
,multi_class,'deprecated'


In [9]:
y_test = test_df["label_id"]
test_pred = final_model.predict(X_test)

print("Test accuracy:", accuracy_score(y_test, test_pred))
print(
  "Test macro F1:",
  f1_score(y_test, test_pred, average="macro", zero_division=0)
)
print(
  "Test weighted F1:",
  f1_score(y_test, test_pred, average="weighted", zero_division=0)
)

print("\nClassification report:")
print(classification_report(y_test, test_pred, zero_division=0))

Test accuracy: 0.3276053215077605
Test macro F1: 0.13102259332707025
Test weighted F1: 0.2962248511033836

Classification report:
              precision    recall  f1-score   support

           0       0.46      0.48      0.47       786
           1       0.16      0.35      0.22        20
           2       0.33      0.01      0.02        88
           3       0.09      0.19      0.12        26
           4       0.25      0.08      0.12        13
           5       0.00      0.00      0.00        21
           6       0.32      0.53      0.40       346
           7       0.18      0.01      0.02       375
           8       0.12      0.09      0.10        57
           9       0.00      0.00      0.00        55
          10       0.03      0.14      0.05         7
          11       0.02      0.40      0.04        10

    accuracy                           0.33      1804
   macro avg       0.16      0.19      0.13      1804
weighted avg       0.32      0.33      0.30      1804



# Added experiment: pretrained Word2Vec + Logistic Regression

The original TF-IDF cells above are unchanged. **This section runs independently:** start with the next setup cell if you only want Word2Vec. The original TF-IDF section references two CSV filenames that are absent from the friend's ZIP, so running every cell from the top will still require those original inputs.

This experiment reads `train_data_with_preprocessing2.csv` and `test_data_with_preprocessing2.csv`. It uses their cleaned English columns and preserves the literal `None` label. It averages frozen 300-dimensional Google News Word2Vec vectors separately for context and reply, normalizes them, concatenates them, then trains Logistic Regression.

The retained file is the existing **6,217-word lookup extracted from genuine Google News Word2Vec**, not the complete three-million-entry model. Missing words contribute no vectors; all-unknown texts receive zeros. Coverage is reported below. No random or locally trained vectors replace missing pretrained words.

For comparability with the original notebook, use its 10-fold `StratifiedGroupKFold` settings and Logistic Regression configuration. The supplied `group` IDs come from the friend's CSV. They do not guarantee that identical context text stays in one fold; the overlap diagnostic makes this limitation visible. Test data never fits a vocabulary, pooling statistic, or classifier.

Dependencies in the notebook kernel: `numpy pandas scipy scikit-learn gensim`. If needed, install missing packages using `%pip install gensim` in a separate cell.

Pretrained source: https://github.com/piskvorky/gensim-data (word2vec-google-news-300). The original archive checksum is `a5e5354d40acb95f9ec66d5977d140ef`.

In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
from gensim.models import KeyedVectors
from sklearn.preprocessing import normalize
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report
from IPython.display import display

# Find the project from either the root or this notebook's folder.
for w2v_folder in [Path.cwd(), *Path.cwd().parents]:
    if (w2v_folder / 'preprocessing/train_data_with_preprocessing2.csv').exists():
        w2v_root = w2v_folder
        break
else:
    raise FileNotFoundError('Open the notebook inside the extracted project folder.')

w2v_train_path = w2v_root / 'preprocessing/train_data_with_preprocessing2.csv'
w2v_test_path = w2v_root / 'preprocessing/test_data_with_preprocessing2.csv'
w2v_vector_path = w2v_root / 'pretrained/google_news_lookup.kv'
w2v_train = pd.read_csv(w2v_train_path, keep_default_na=False)
w2v_test = pd.read_csv(w2v_test_path, keep_default_na=False)
w2v_vectors = KeyedVectors.load(str(w2v_vector_path), mmap='r')
assert w2v_vectors.vector_size == 300

w2v_label_map = dict(zip(w2v_train.label_id, w2v_train.label))
w2v_label_ids = sorted(w2v_label_map)
assert w2v_label_ids == list(range(12)) and w2v_label_map[0] == 'None'
for w2v_frame in [w2v_train, w2v_test]:
    assert (w2v_frame.label_id.map(w2v_label_map) == w2v_frame.label).all()
    assert ((w2v_frame.label == 'None') == (w2v_frame.is_toxic == 0)).all()
print('Train rows:', len(w2v_train), '| Test rows:', len(w2v_test))
print('Frozen lookup:', len(w2v_vectors), 'words;', w2v_vectors.vector_size, 'dimensions')


Train rows: 19431 | Test rows: 1804
Frozen lookup: 6217 words; 300 dimensions


In [2]:
def w2v_sentence_vector(text):
    words = [word for word in str(text).split() if word in w2v_vectors]
    if not words:
        return np.zeros(w2v_vectors.vector_size, dtype=np.float32)
    return w2v_vectors[words].mean(axis=0)


def w2v_features(contexts, replies):
    context_vectors = normalize(np.asarray([w2v_sentence_vector(text) for text in contexts], dtype=np.float32))
    reply_vectors = normalize(np.asarray([w2v_sentence_vector(text) for text in replies], dtype=np.float32))
    return normalize(np.hstack([context_vectors, reply_vectors]))


# Pooling uses fixed external vectors only, so it can be reused across folds.
w2v_X_train = w2v_features(w2v_train.prev_clean, w2v_train.target_clean)
w2v_X_test = w2v_features(w2v_test.context_clean, w2v_test.reply_clean)
w2v_y_train = w2v_train.label_id.to_numpy()
w2v_y_test = w2v_test.label_id.to_numpy()
assert w2v_X_train.shape == (len(w2v_train), 600)
assert w2v_X_test.shape == (len(w2v_test), 600)
assert np.isfinite(w2v_X_train).all() and np.isfinite(w2v_X_test).all()
print('Training features:', w2v_X_train.shape, '| Test features:', w2v_X_test.shape)

w2v_coverage_rows = []
for partition, frame, columns in [
    ('train', w2v_train, ['prev_clean', 'target_clean']),
    ('test', w2v_test, ['context_clean', 'reply_clean']),
]:
    for column in columns:
        sentences = [str(text).split() for text in frame[column]]
        token_count = sum(len(words) for words in sentences)
        matched = sum(word in w2v_vectors for words in sentences for word in words)
        w2v_coverage_rows.append(dict(
            partition=partition, field=column,
            token_coverage=matched / max(token_count, 1),
            all_unknown_rows=sum(not any(word in w2v_vectors for word in words) for words in sentences),
        ))
w2v_coverage = pd.DataFrame(w2v_coverage_rows)
display(w2v_coverage.round(4))


Training features: (19431, 600) | Test features: (1804, 600)


,partition,field,token_coverage,all_unknown_rows
0,train,prev_clean,0.9771,0
1,train,target_clean,0.9926,0
2,test,context_clean,0.9498,1
3,test,reply_clean,0.9400,7


In [3]:
def w2v_new_classifier():
    return LogisticRegression(
        C=1.0, class_weight='balanced', max_iter=2000,
        solver='lbfgs', random_state=42,
    )


w2v_splitter = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)
w2v_cv_rows = []
for w2v_fold, (w2v_fit_indices, w2v_val_indices) in enumerate(w2v_splitter.split(
    w2v_X_train, w2v_y_train, groups=w2v_train.group,
)):
    w2v_fold_model = w2v_new_classifier()
    w2v_fold_model.fit(w2v_X_train[w2v_fit_indices], w2v_y_train[w2v_fit_indices])
    w2v_val_pred = w2v_fold_model.predict(w2v_X_train[w2v_val_indices])
    w2v_fit_contexts = set(w2v_train.iloc[w2v_fit_indices].prev_clean)
    w2v_val_contexts = set(w2v_train.iloc[w2v_val_indices].prev_clean)
    w2v_cv_rows.append(dict(
        fold=w2v_fold, train_rows=len(w2v_fit_indices), validation_rows=len(w2v_val_indices),
        macro_f1=f1_score(w2v_y_train[w2v_val_indices], w2v_val_pred, labels=w2v_label_ids, average='macro', zero_division=0),
        weighted_f1=f1_score(w2v_y_train[w2v_val_indices], w2v_val_pred, labels=w2v_label_ids, average='weighted', zero_division=0),
        accuracy=accuracy_score(w2v_y_train[w2v_val_indices], w2v_val_pred),
        shared_context_types=len(w2v_fit_contexts & w2v_val_contexts),
    ))
    print('Fold', w2v_fold, '| macro F1:', round(w2v_cv_rows[-1]['macro_f1'], 4), flush=True)
w2v_cv_results = pd.DataFrame(w2v_cv_rows)
display(w2v_cv_results.round(4))
display(w2v_cv_results[['macro_f1', 'weighted_f1', 'accuracy']].agg(['mean', 'std']).round(4))


Fold 0 | macro F1: 0.9456
Fold 1 | macro F1: 0.9477
Fold 2 | macro F1: 0.939
Fold 3 | macro F1: 0.9309
Fold 4 | macro F1: 0.9446
Fold 5 | macro F1: 0.9315
Fold 6 | macro F1: 0.9504
Fold 7 | macro F1: 0.9397
Fold 8 | macro F1: 0.9304
Fold 9 | macro F1: 0.9354


,fold,train_rows,validation_rows,macro_f1,weighted_f1,accuracy,shared_context_types
0,0,17487,1944,0.9456,0.9618,0.9614,839
1,1,17488,1943,0.9477,0.9639,0.9635,836
2,2,17488,1943,0.9390,0.9563,0.9563,843
3,3,17488,1943,0.9309,0.9515,0.9516,804
4,4,17488,1943,0.9446,0.9567,0.9568,855
5,5,17488,1943,0.9315,0.9491,0.9490,827
6,6,17488,1943,0.9504,0.9647,0.9645,845
7,7,17488,1943,0.9397,0.9563,0.9557,870
8,8,17488,1943,0.9304,0.9455,0.9449,849
9,9,17488,1943,0.9354,0.9528,0.9521,828


,macro_f1,weighted_f1,accuracy
mean,0.9395,0.9559,0.9556
std,0.0073,0.0063,0.0063


In [4]:
# Fit the frozen configuration on all training rows, then evaluate once.
w2v_final_model = w2v_new_classifier()
w2v_final_model.fit(w2v_X_train, w2v_y_train)
w2v_test_pred = w2v_final_model.predict(w2v_X_test)
w2v_test_results = dict(
    task='12-class subtype', test_rows=len(w2v_test),
    accuracy=float(accuracy_score(w2v_y_test, w2v_test_pred)),
    macro_f1=float(f1_score(w2v_y_test, w2v_test_pred, labels=w2v_label_ids, average='macro', zero_division=0)),
    weighted_f1=float(f1_score(w2v_y_test, w2v_test_pred, labels=w2v_label_ids, average='weighted', zero_division=0)),
)
display(pd.DataFrame([w2v_test_results]).round(4))
print(classification_report(
    w2v_y_test, w2v_test_pred, labels=w2v_label_ids,
    target_names=[w2v_label_map[label] for label in w2v_label_ids], zero_division=0,
))


                                     precision    recall  f1-score   support

                               None       0.51      0.31      0.38       786
                     Dehumanization       0.05      0.35      0.09        20
                         Harassment       0.00      0.00      0.00        88
                        Hate Speech       0.08      0.38      0.13        26
                    Identity Attack       0.08      0.31      0.13        13
 Incitement / Encouragement of Harm       0.06      0.10      0.08        21
                             Insult       0.32      0.22      0.26       346
                 Mockery / Ridicule       0.34      0.05      0.09       375
                          Profanity       0.05      0.14      0.08        57
Sexual Harassment / Objectification       0.04      0.11      0.06        55
                             Threat       0.04      0.29      0.07         7
      Toxic Endorsement / Agreement       0.01      0.20      0.01        1

,task,test_rows,accuracy,macro_f1,weighted_f1
0,12-class subtype,1804,0.2095,0.1151,0.2447


## Interpretation

These are new Word2Vec results on the supplied preprocessing2 files. The original TF-IDF outputs above remain historical and use different referenced filenames; this is not a controlled TF-IDF-versus-Word2Vec comparison. Accuracy alone can hide poor recognition of rare toxicity subtypes, so inspect macro F1 and the class report.

The frozen lookup has limited vocabulary. The provided grouping can share contexts across folds, and internal cross-validation performance may not transfer to real test comments. No neural network, CNN, pretrained transformer or newly trained embedding is used in this section.

# Binary toxicity: pretrained Word2Vec + Logistic Regression

Predict `0 = Non-toxic` and `1 = Toxic` using the supplied `is_toxic` labels. Run the Word2Vec setup and feature cells above first. This section reuses their exact 600-dimensional context-and-reply features and the same Logistic Regression settings. It trains a separate binary classifier, rather than converting the 12-class predictions into binary labels.

The ten validation folds are identical to the 12-class Word2Vec experiment: they use its subtype-stratified grouped split with seed 42. The supplied groups can share context text across folds, so the earlier validation limitation still applies. Final evaluation uses all 1,804 prepared test rows, with no test-based parameter tuning.

In [2]:
w2v_binary_y_train = w2v_train.is_toxic.to_numpy(dtype=np.int64)
w2v_binary_y_test = w2v_test.is_toxic.to_numpy(dtype=np.int64)
assert set(w2v_binary_y_train) == {0, 1}
assert np.array_equal(w2v_binary_y_train, (w2v_y_train != 0).astype(np.int64))
assert np.array_equal(w2v_binary_y_test, (w2v_y_test != 0).astype(np.int64))

w2v_binary_cv_rows = []
w2v_binary_splitter = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)
# Keep precisely the same folds as the 12-class Word2Vec experiment.
for w2v_binary_fold, (w2v_binary_fit, w2v_binary_val) in enumerate(w2v_binary_splitter.split(
    w2v_X_train, w2v_y_train, groups=w2v_train.group,
)):
    w2v_binary_fold_model = w2v_new_classifier()
    w2v_binary_fold_model.fit(w2v_X_train[w2v_binary_fit], w2v_binary_y_train[w2v_binary_fit])
    w2v_binary_val_pred = w2v_binary_fold_model.predict(w2v_X_train[w2v_binary_val])
    w2v_binary_cv_rows.append(dict(
        fold=w2v_binary_fold,
        train_rows=len(w2v_binary_fit), validation_rows=len(w2v_binary_val),
        accuracy=float(accuracy_score(w2v_binary_y_train[w2v_binary_val], w2v_binary_val_pred)),
        macro_f1=float(f1_score(w2v_binary_y_train[w2v_binary_val], w2v_binary_val_pred, labels=[0, 1], average='macro', zero_division=0)),
        weighted_f1=float(f1_score(w2v_binary_y_train[w2v_binary_val], w2v_binary_val_pred, labels=[0, 1], average='weighted', zero_division=0)),
    ))
    print('Binary fold', w2v_binary_fold, '| macro F1:', round(w2v_binary_cv_rows[-1]['macro_f1'], 4), flush=True)
w2v_binary_cv_results = pd.DataFrame(w2v_binary_cv_rows)
display(w2v_binary_cv_results.round(4))
display(w2v_binary_cv_results[['accuracy', 'macro_f1', 'weighted_f1']].agg(['mean', 'std']).round(4))


Binary fold 0 | macro F1: 0.9877
Binary fold 1 | macro F1: 0.9774
Binary fold 2 | macro F1: 0.9753
Binary fold 3 | macro F1: 0.9763
Binary fold 4 | macro F1: 0.9789
Binary fold 5 | macro F1: 0.9737
Binary fold 6 | macro F1: 0.9799
Binary fold 7 | macro F1: 0.9758
Binary fold 8 | macro F1: 0.9655
Binary fold 9 | macro F1: 0.9768


,fold,train_rows,validation_rows,accuracy,macro_f1,weighted_f1
0,0,17487,1944,0.9877,0.9877,0.9877
1,1,17488,1943,0.9774,0.9774,0.9774
2,2,17488,1943,0.9753,0.9753,0.9753
3,3,17488,1943,0.9763,0.9763,0.9763
4,4,17488,1943,0.9789,0.9789,0.9789
5,5,17488,1943,0.9738,0.9737,0.9738
6,6,17488,1943,0.9799,0.9799,0.9799
7,7,17488,1943,0.9758,0.9758,0.9758
8,8,17488,1943,0.9655,0.9655,0.9655
9,9,17488,1943,0.9768,0.9768,0.9768


,accuracy,macro_f1,weighted_f1
mean,0.9767,0.9767,0.9767
std,0.0055,0.0055,0.0055


In [3]:
# Fit a new binary model on all training rows with the frozen settings.
w2v_binary_final_model = w2v_new_classifier()
w2v_binary_final_model.fit(w2v_X_train, w2v_binary_y_train)
w2v_binary_test_pred = w2v_binary_final_model.predict(w2v_X_test)
w2v_binary_test_results = dict(
    task='binary toxicity', test_rows=len(w2v_test),
    accuracy=float(accuracy_score(w2v_binary_y_test, w2v_binary_test_pred)),
    macro_f1=float(f1_score(w2v_binary_y_test, w2v_binary_test_pred, labels=[0, 1], average='macro', zero_division=0)),
    weighted_f1=float(f1_score(w2v_binary_y_test, w2v_binary_test_pred, labels=[0, 1], average='weighted', zero_division=0)),
)
display(pd.DataFrame([w2v_binary_test_results]).round(4))
print(classification_report(
    w2v_binary_y_test, w2v_binary_test_pred,
    labels=[0, 1], target_names=['Non-toxic', 'Toxic'], zero_division=0,
))


              precision    recall  f1-score   support

   Non-toxic       0.62      0.25      0.36       786
       Toxic       0.60      0.88      0.72      1018

    accuracy                           0.61      1804
   macro avg       0.61      0.57      0.54      1804
weighted avg       0.61      0.61      0.56      1804



,task,test_rows,accuracy,macro_f1,weighted_f1
0,binary toxicity,1804,0.607,0.538,0.561
